<a href="https://colab.research.google.com/github/smagadi/AIML/blob/master/GenAI/LightweightFineTuning_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique:
* Model:
* Evaluation approach:
* Fine-tuning dataset:

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [ ]:
!pip install scikit-learn
!pip install datasets
!pip install accelerate
!pip install transformers datasets evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 81.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 35.1 MB/s eta 0:00:00
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 942.2 kB/s eta 0:00:000:01:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/669M [00:00<?, ?B/s]

In [ ]:
dataset = load_dataset("yelp_review_full")
small_train_ds = dataset["train"].shuffle(seed=42).select(range(1000))
small_test_ds = dataset["test"].shuffle(seed=42).select(range(1000))
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, return_tensors="pt")
tok_train_dataset = small_train_ds.map(tokenize_function, batched=True)
tok_test_dataset = small_test_ds.map(tokenize_function, batched=True)

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
import evaluate


metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
print(model)



BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
#!pip install --upgrade transformers
#!pip install --upgrade accelerate
#!pip install transformers[torch]
import accelerate
training_args = TrainingArguments(
    output_dir="./my_awesome_model_org",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train_dataset,
    eval_dataset=tok_test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.982415,0.601000
2,No log,1.006377,0.618000


Checkpoint destination directory ./my_awesome_model_org/checkpoint-125 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./my_awesome_model_org/checkpoint-250 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=250, training_loss=0.6065675048828125, metrics={'train_runtime': 288.2193, 'train_samples_per_second': 6.939, 'train_steps_per_second': 0.867, 'total_flos': 526236284928000.0, 'train_loss': 0.6065675048828125, 'epoch': 2.0})

## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [ ]:
!pip install --upgrade peft
from peft import LoraConfig, PeftModel

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType


lora_config = LoraConfig(
 r=16,
 lora_alpha=32,
 target_modules=["query", "value"],
 lora_dropout=0.05,
 bias="none",
 task_type=TaskType.SEQ_CLS, # this is necessary
 inference_mode=False
)


pft_model_new = get_peft_model(model, lora_config)


In [ ]:
pft_model_new.print_trainable_parameters()

pft_model_new

trainable params: 589,824 || all params: 167,953,930 || trainable%: 0.35118201759256246


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(105879, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): Linear(
                    in_features=768, out_features=768, bias=True
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, ou

In [ ]:

training_args = TrainingArguments(output_dir="bert_peft_trainer01")
bert_peft_trainer = Trainer(
    model=pft_model_new,
    args=training_args,
    train_dataset=tok_train_dataset,
    eval_dataset=tok_test_dataset,
    compute_metrics=compute_metrics,
)
bert_peft_trainer.train()

Step,Training Loss


TrainOutput(global_step=375, training_loss=0.45441463216145833, metrics={'train_runtime': 221.5844, 'train_samples_per_second': 13.539, 'train_steps_per_second': 1.692, 'total_flos': 794825680896000.0, 'train_loss': 0.45441463216145833, 'epoch': 3.0})

In [ ]:
### chnage the LORA params again , removed Value from the target modules
## R= 8
## loar_alpha=64
from peft import LoraConfig, get_peft_model, TaskType


lora_config02 = LoraConfig(
 r=16,
 lora_alpha=64,
 target_modules=[ "value"],
 lora_dropout=0.05,
 bias="none",
 task_type=TaskType.SEQ_CLS,
 inference_mode=False
)


pft_model02 = get_peft_model(model, lora_config02)

In [ ]:
pft_model02.print_trainable_parameters()

pft_model02

trainable params: 589,824 || all params: 167,953,930 || trainable%: 0.35118201759256246


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(105879, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): Linear(
                    in_features=768, out_features=768, bias=True
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, ou

In [ ]:
bert_peft_trainer002 = Trainer(
    model=pft_model02,
    args=training_args,
    train_dataset=tok_train_dataset,
    eval_dataset=tok_test_dataset,
    compute_metrics=compute_metrics,
)


In [ ]:
bert_peft_trainer002.train()

Step,Training Loss


TrainOutput(global_step=375, training_loss=0.4319875895182292, metrics={'train_runtime': 209.4569, 'train_samples_per_second': 14.323, 'train_steps_per_second': 1.79, 'total_flos': 794825680896000.0, 'train_loss': 0.4319875895182292, 'epoch': 3.0})

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [ ]:
# Orginal Pre Trained Model
trainer.evaluate()

{'eval_loss': 1.0112253427505493,
 'eval_accuracy': 0.618,
 'eval_runtime': 35.59,
 'eval_samples_per_second': 28.098,
 'eval_steps_per_second': 3.512,
 'epoch': 2.0}

In [ ]:
#peft Trained one
bert_peft_trainer.evaluate()

{'eval_loss': 1.0112253427505493,
 'eval_accuracy': 0.618,
 'eval_runtime': 34.1953,
 'eval_samples_per_second': 29.244,
 'eval_steps_per_second': 3.655,
 'epoch': 3.0}

In [ ]:
#peft 02 Trained one
bert_peft_trainer002.evaluate()

In [ ]:
### Conlcusion : Not much has improved .  ...

In [ ]:
## Attribution to the following
## i have used the foloowing 1.Gemini to undesrtand the code 2.Your own chat
## https://jaotheboss.medium.com/peft-with-bert-8763d8b8a4ca : This helped me tunderstand even better